# 07b — Train & Evaluate (Bootstrap branch)

Parallel branch of `07_train_eval.ipynb`. Same scenario coverage (A-E x
{linear, mlp2}, B-E ablation variants, G/XGBoost baseline), but replaces
repeated spatial k-fold with bootstrap resampling: each repeat samples
the full dataset WITH REPLACEMENT, dedups by point_id (so train/val/test
stay disjoint by point identity), then splits by percentage.

**This notebook does NOT run the formal Wilcoxon / Nadeau-Bengio
significance tests from 07.** Those tests are specifically designed for
paired, correlated scores from repeated k-fold CV — bootstrap-repeat
scores aren't the same statistical object (no partition structure, no
guaranteed one-test-per-point-per-repeat), so applying those tests here
would be statistically questionable. This notebook reports descriptive
aggregates (mean +/- std across repeats) only. If you want formal
significance testing on bootstrap results later, that needs its own
design (e.g. bootstrap confidence intervals), not the pre-specified
k-fold pairs/tests.

Also uses larger model capacity (`fusion_dim`, `hidden_dim`) with
increased regularization (`dropout`, `weight_decay`), per the diagnostic
against `07`'s overfitting pattern (val PR-AUC plateaued from ~epoch 2-3
while train loss kept falling).

GPU recommended, not required (graphs are small).

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# TEMP: install locally patched train.py / models.py / plot_history.py
# until pushed to GitHub. Skip this cell once the repo itself is updated.
from google.colab import files
import shutil

print("Upload train.py, models.py, and plot_history.py:")
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f"{REPO_DIR}/src/{fname}")
print("Patched files installed:", list(uploaded.keys()))

In [ ]:
!pip install -q torch_geometric xgboost scikit-learn scipy pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path
import torch

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/eval_bootstrap.yaml") as f:
    eval_cfg = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/model_bootstrap.yaml") as f:
    model_cfg = yaml.safe_load(f)

PROCESSED_DIR = Path(paths_cfg["processed_dir"])
OUTPUTS_DIR = Path(paths_cfg["outputs_dir"])
# separate checkpoint/metrics dirs from 07 -- keeps bootstrap results from
# colliding with or overwriting the formal k-fold pipeline's outputs
CHECKPOINT_DIR = OUTPUTS_DIR / "checkpoints_bootstrap"
METRICS_DIR = OUTPUTS_DIR / "metrics_bootstrap"
for d in [CHECKPOINT_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
config = {"batch_size": eval_cfg.get("batch_size") or 32,
          "epoch_cap": eval_cfg.get("epoch_cap") or 150,
          "patience": eval_cfg.get("patience") or 20,
          "lr_patience": eval_cfg.get("lr_patience") or 5,
          "lr": eval_cfg.get("lr") or 5e-3,
          "weight_decay": eval_cfg.get("weight_decay") or 1e-4,
          "fusion_dim": model_cfg.get("fusion_dim") or 128,
          "val_frac": eval_cfg.get("val_frac") or 0.15,
          "test_frac": eval_cfg.get("test_frac") or 0.15,
          "num_workers": eval_cfg.get("num_workers") or 0,
          "use_amp": eval_cfg.get("use_amp", True)}
N_REPEATS = eval_cfg.get("n_repeats") or 20
print(f"Device: {device} | config: {config} | n_repeats: {N_REPEATS}")

In [ ]:
import pandas as pd
import graph_datasets as ds
import train as tr
import evaluate as ev
import models

index_df = pd.read_parquet(PROCESSED_DIR / "dataset_index.parquet")
dataset = ds.DualGraphDataset(index_df, PROCESSED_DIR / "svg_graphs", PROCESSED_DIR / "tvg_graphs")
print(f"Dataset: {len(dataset)} points (bootstrap branch -- no fold_cols needed)")

svg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("svg_layers", 2), dropout=model_cfg.get("dropout", 0.45),
                   signage_vocab=5, light_pole_vocab=4, road_marking_vocab=2, cat_embed_dim=2)
tvg_kwargs = dict(hidden_dim=model_cfg.get("hidden_dim", 128), heads=model_cfg.get("heads", 4),
                   num_layers=model_cfg.get("tvg_layers", 2), dropout=model_cfg.get("dropout", 0.45),
                   building_type_vocab=58, highway_vocab=13,
                   building_type_embed_dim=8, highway_embed_dim=4)

In [ ]:
# ── Train every primary scenario x head depth ─────────────────────────
# Split into one cell per scenario below, same rationale as 07: run /
# monitor / interrupt independently rather than one long nested loop.
PRIMARY_SCENARIOS = ["A", "B", "C", "D", "E"]
HEAD_DEPTHS = ["linear", "mlp2"]
all_results = {}

### Scenario A — SVG only

In [ ]:
# ── Scenario A: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"A_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_bootstrap("A", depth, use_ablation=False, dataset=dataset,
                                         n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                         tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario B — TVG only

In [ ]:
# ── Scenario B: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"B_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_bootstrap("B", depth, use_ablation=False, dataset=dataset,
                                         n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                         tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario C — dual graph (concat)

In [ ]:
# ── Scenario C: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"C_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_bootstrap("C", depth, use_ablation=False, dataset=dataset,
                                         n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                         tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario D — dual graph (late fusion)

In [ ]:
# ── Scenario D: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"D_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_bootstrap("D", depth, use_ablation=False, dataset=dataset,
                                         n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                         tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Scenario E — dual graph (cross-attention)

In [ ]:
# ── Scenario E: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"E_{depth}"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_bootstrap("E", depth, use_ablation=False, dataset=dataset,
                                         n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                         tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

In [ ]:
# ── Ablation: B-E only (F deferred) ────────────────────────────────────
# Split into one cell per scenario, same rationale as the primary block above.

### Ablation B+

In [ ]:
# ── Scenario B ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"B_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_bootstrap("B", depth, use_ablation=True, dataset=dataset,
                                         n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                         tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Ablation C+

In [ ]:
# ── Scenario C ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"C_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_bootstrap("C", depth, use_ablation=True, dataset=dataset,
                                         n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                         tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Ablation D+

In [ ]:
# ── Scenario D ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"D_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_bootstrap("D", depth, use_ablation=True, dataset=dataset,
                                         n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                         tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

### Ablation E+

In [ ]:
# ── Scenario E ablation: linear + mlp2 ─────────────────────────────────
for depth in HEAD_DEPTHS:
    key = f"E_{depth}_ablation"
    print(f"\n=== {key} ===")
    results = tr.run_scenario_bootstrap("E", depth, use_ablation=True, dataset=dataset,
                                         n_repeats=N_REPEATS, config=config, svg_kwargs=svg_kwargs,
                                         tvg_kwargs=tvg_kwargs, device=device, checkpoint_dir=CHECKPOINT_DIR)
    all_results[key] = results
    print(f"  {len(results)} repeat-runs complete.")

In [ ]:
# ── Scenario G: XGBoost, separate path (bootstrap version) ────────────
import baseline_features
from xgboost import XGBClassifier
import numpy as np

feat_table = baseline_features.build_feature_table(index_df["point_id"].tolist(),
                                                     PROCESSED_DIR / "svg_graphs", PROCESSED_DIR / "tvg_graphs", torch)
feat_table = feat_table.merge(index_df[["point_id", "label"]], on="point_id")
feature_cols = [c for c in feat_table.columns if c not in ["point_id", "label"]]

n_total = len(feat_table)
g_results = []
for repeat in range(N_REPEATS):
    rng = np.random.default_rng(42 + repeat)
    boot_idx = rng.integers(0, n_total, size=n_total)
    boot_df = feat_table.iloc[boot_idx].drop_duplicates(subset="point_id", keep="first")
    shuffled = boot_df.sample(frac=1.0, random_state=42 + repeat)

    n = len(shuffled)
    n_val = int(round(n * config["val_frac"]))
    n_test = int(round(n * config["test_frac"]))
    val_ = shuffled.iloc[:n_val]
    test = shuffled.iloc[n_val:n_val + n_test]
    train_ = shuffled.iloc[n_val + n_test:]

    clf = XGBClassifier(n_estimators=200, max_depth=4, eval_metric="aucpr", random_state=42)
    clf.fit(train_[feature_cols], train_["label"])
    prob = clf.predict_proba(test[feature_cols])[:, 1]
    g_results.append({"repeat": repeat, "n_train": len(train_), "n_val": len(val_), "n_test": len(test),
                       **ev.compute_metrics(test["label"].values, prob)})

all_results["G"] = g_results
print(f"Scenario G: {len(g_results)} repeat-runs complete.")

In [ ]:
# ── Aggregate + report EVERY scenario, regardless of performance ────────
agg_rows = []
for key, results in all_results.items():
    agg = ev.aggregate_fold_results(results)   # works identically on bootstrap-repeat dicts
    row = {"scenario": key}
    for metric, (mean, std) in agg.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(METRICS_DIR / "all_scenarios_summary_bootstrap.csv", index=False)
display(agg_df)

In [ ]:
# ── Head-depth comparison: descriptive only (no formal significance test
#    here -- see notebook intro for why). Pick a winner per scenario using
#    mean PR-AUC, same as 07, for consistency if you want ONE head depth
#    reported per scenario. ──
depth_compare = []
for scenario in PRIMARY_SCENARIOS:
    lin = agg_df[agg_df["scenario"] == f"{scenario}_linear"]["pr_auc_mean"].iloc[0]
    mlp = agg_df[agg_df["scenario"] == f"{scenario}_mlp2"]["pr_auc_mean"].iloc[0]
    depth_compare.append({"scenario": scenario, "linear_pr_auc": lin, "mlp2_pr_auc": mlp})

depth_df = pd.DataFrame(depth_compare)
display(depth_df)

WINNING_DEPTH = "linear" if depth_df["linear_pr_auc"].mean() >= depth_df["mlp2_pr_auc"].mean() else "mlp2"
print(f"\nWinning head depth (by mean PR-AUC across scenarios): {WINNING_DEPTH}")
print("Descriptive only -- no Wilcoxon/Nadeau-Bengio run in this branch (see intro).")

## Epoch-level diagnostics

Per-repeat training history (train loss, val PR-AUC, val AUROC per epoch)
is saved under `CHECKPOINT_DIR/{tag}_history/repeat{N}.json`, same JSON
shape as 07's per-fold history files. **Note:** `plot_history.py`'s
`plot_fold`/`plot_all_folds` (used in 07) assume a `{fold_col}_fold{id}`
naming pattern, not `repeat{N}` -- I haven't seen plot_history.py's
actual code, so I can't confirm whether it already has a path-based entry
point that works with these filenames. Load the JSON directly below as a
safe fallback; if plot_history.py turns out to already support arbitrary
paths, swap this for that instead.

In [ ]:
import json
import matplotlib.pyplot as plt

# Direct fallback: load one repeat's history and plot train_loss / val_pr_auc
# / val_auroc, with the best (checkpoint-selected) epoch marked -- doesn't
# assume any particular helper exists in plot_history.py.
history_path = CHECKPOINT_DIR / "A_linear_history" / "repeat0.json"
history = json.loads(history_path.read_text())

epochs = [h["epoch"] for h in history]
best_epoch = max(range(len(history)), key=lambda i: history[i]["val_pr_auc"])

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(epochs, [h["train_loss"] for h in history], label="train_loss", color="tab:blue")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train_loss", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(epochs, [h["val_pr_auc"] for h in history], label="val_pr_auc", color="tab:orange")
ax2.plot(epochs, [h["val_auroc"] for h in history], label="val_auroc", color="tab:green")
ax2.axvline(best_epoch, color="gray", linestyle="--", label=f"best epoch ({best_epoch})")
ax2.set_ylabel("val metric")

fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
plt.title(f"repeat0 history ({history_path.name})")
plt.tight_layout()
plt.show()

In [ ]:
print("Bootstrap branch complete.")
print("For formal significance testing, compare against 07_train_eval.ipynb's k-fold results directly,")
print("or design bootstrap-specific confidence intervals separately -- not the k-fold Wilcoxon/Nadeau-Bengio pair.")